# Notebook 3 — Forecasting, Portfolio Optimization, and Backtesting for Fantasy Premier League

**Thesis Pipeline — Final Stage**

---

## Abstract

This notebook presents the modeling, optimization, and evaluation components of a Fantasy Premier League (FPL) forecasting system trained on six seasons of official match data (2020-21 through 2025-26). Building on the identity-resolved, feature-engineered master training set produced by the preceding notebooks in this pipeline, we formulate three complementary prediction tasks: (i) single-gameweek point regression, (ii) binary haul classification, and (iii) three-gameweek cumulative point regression.

A family of seventeen gradient-boosted tree models is developed across these tasks, spanning a progression from baseline regression through ranking-informed blending, reliability-weighted regularisation, Dropout Additive Regression Trees (DART) with elite-player sample weighting, xG/xA-enriched variants, and hyperparameter-optimised Safe/Aggressive specialists produced via Tree-structured Parzen Estimator (TPE) search [^optuna]. Every model is evaluated on a fixed-window temporal holdout with stratified error analysis by scoring tier, providing calibrated risk-reward profiles for downstream decision-making.

Two portfolio construction engines consume the predictive outputs: a relief-score heuristic optimizer (the production engine) and a mixed-integer linear programming (MILP) formulation of the Markowitz mean-variance problem [^markowitz] solved via CBC [^cbc]. Both respect the full FPL rule set — budget constraints, position quotas, per-club limits, and formation legality — and produce captaincy decisions that integrate with a binary haul classifier to balance expected-value optimisation against ceiling detection.

A ten-gameweek backtest simulation validates the optimisation pipeline end-to-end, including FPL automatic substitution rules, captain-did-not-play vice-captain replacement, and the formation constraints that govern legal substitution events. Results are reported with realistic calibration factors empirically derived from historical over-prediction bias, providing honest upper bounds for expected squad performance rather than the inflated point totals a naïve interpretation of regression output would suggest.

The final deliverable integrates all predictive components, optimisation engines, and auxiliary tooling (availability filtering, name-collision resolution, dynamic double/blank gameweek detection, price-change tracking) into a deployed application layer of interactive squad-building, transfer-recommendation, captaincy-selection, and backtesting interfaces. All seventeen models are retained and reported in a master audit ledger; the eight core models receive full methodological treatment and the nine ablation models are documented in a dedicated ablation studies section.

---

## Keywords

Fantasy Premier League, gradient boosting, LightGBM, LambdaRank, DART, binary classification, Optuna hyperparameter optimisation, mean-variance portfolio optimisation, mixed-integer linear programming, temporal cross-validation, stratified error analysis, automatic substitution, captaincy decision theory

---

[^optuna]: T. Akiba *et al.*, "Optuna: A Next-generation Hyperparameter Optimization Framework," in *Proc. 25th ACM SIGKDD Int. Conf. on Knowledge Discovery & Data Mining*, 2019, pp. 2623–2631, doi: 10.1145/3292500.3330701.

[^markowitz]: H. Markowitz, "Portfolio Selection," *The Journal of Finance*, vol. 7, no. 1, pp. 77–91, Mar. 1952, doi: 10.2307/2975974.

[^cbc]: J. Forrest and R. Lougee-Heimer, "CBC User Guide," in *Emerging Theory, Methods, and Applications*, H. J. Greenberg and J. C. Smith, Eds. INFORMS, 2005, doi: 10.1287/educ.1053.0020.

## 1. Introduction and Objectives

### 1.1 Problem Statement

Fantasy Premier League is a weekly resource-allocation problem under uncertainty. Each manager selects a fifteen-player squad subject to a £100.0m budget, position quotas (2 goalkeepers, 5 defenders, 5 midfielders, 3 forwards), a maximum of three players per Premier League club, and the requirement to nominate a starting eleven in a legal formation each gameweek. Players accumulate points based on in-match events (goals, assists, clean sheets, bonus points, minutes played) with position-dependent scoring, and managers receive double points from their nominated captain.

Three decisions drive weekly performance: the composition of the fifteen-player squad (updated via weekly transfers, each beyond the first costing a four-point penalty), the selection of a legal starting eleven from that squad, and the nomination of captain and vice-captain (the vice-captain's points double if the captain plays zero minutes). The underlying prediction problem is therefore not single-valued — different squad decisions have different optimal predictive targets, and a model that minimises mean absolute error on the full squad may systematically underestimate the players most relevant for captaincy.

### 1.2 Research Questions

This notebook addresses four research questions:

**RQ1** — *How does model architecture affect predictive accuracy across the scoring distribution?* Specifically, does a regression model trained to minimise mean absolute error (MAE) systematically underpredict high-scoring events, and can specialist architectures (ranking, classification, DART with sample weighting) recover this ceiling?

**RQ2** — *What is the optimal lookback window for point prediction?* Shorter windows respond to recent form; longer windows average out noise. We test 4-, 6-, and 8-gameweek windows under matched hyperparameters.

**RQ3** — *Does incorporating underlying expected-goal metrics (xG, xA) improve prediction?* Raw points are a noisy function of shot quality and chance creation; aggregated expected-goal-involvement (xGI) features provide a less volatile signal of attacking contribution.

**RQ4** — *How do heuristic and mathematically optimal portfolio construction methods compare?* The heuristic production optimizer is fast and rule-rich; the MILP solver produces provably optimal allocations. We compare their squads, captaincy decisions, and backtest performance across a ten-gameweek historical window.

### 1.3 Contributions

1. **A seventeen-model comparative study** of gradient-boosted tree architectures for FPL point prediction, reported through a master audit ledger with stratified MAE analysis by scoring tier.

2. **A dual-engine captaincy algorithm** that combines continuous point-regression expected value with binary haul-classification probability, explicitly separating floor estimation from ceiling detection.

3. **A production-grade FPL portfolio optimizer** with full rule compliance: budget, position quotas, per-club limits, formation legality in the starting eleven, and dead-fodder management for bench players with low rotation risk.

4. **A Markowitz mean-variance MILP formulation** for FPL squad selection, producing an efficient frontier across three risk-aversion regimes (Safe / Balanced / Aggressive) with mathematically certified optimality.

5. **A ten-gameweek backtest simulation** including FPL automatic substitution rules and captain-DNP vice-captain replacement, validating the optimisation pipeline against actual historical points.

6. **An empirically calibrated expectation bridge** translating raw model expected-point output into realistic performance ranges, addressing the systematic ~28–38% over-prediction bias inherent to regression on noisy sports data.

### 1.4 Notebook Scope

This notebook is the third and final stage of the thesis pipeline. It consumes the identity-resolved, feature-engineered dataset `master_training_set_no0min.csv` produced by the preceding notebooks and does not re-compute base features. All feature derivations introduced here (windowed lags, 3-gameweek fixture difficulty horizon, points volatility, minutes stability, team point share, expected goal involvement rolling averages, seasonal baselines, and within-season expanding average against difficulty) are modeling-specific transformations of already-validated upstream columns.

## 2. Methodology

The methodology proceeds in eight parts: target variable formalisation (§2.1), feature taxonomy (§2.2), temporal cross-validation strategy (§2.3), core model specifications (§2.4), ablation model specifications (§2.5), portfolio optimisation frameworks (§2.6), auto-substitution and captaincy algorithms (§2.7), and evaluation metrics (§2.8).

The remainder of §2 presents formal specifications. Implementation code appears in §4 and onward; methodology here is deliberately decoupled from code to support defense scrutiny of modelling decisions independent of any particular software realisation.

### 2.1 Target Variable Formalisation

Let $\mathcal{P}$ denote the set of players appearing in the training data and $\mathcal{G}$ the set of gameweeks indexed by $(s, g)$ where $s$ is the season identifier and $g \in \{1, \ldots, 38\}$. For a player $p \in \mathcal{P}$ in gameweek $(s, g)$, let $y_{p,s,g}$ denote the Total Points scored.

Three target variables are defined, corresponding to three distinct prediction tasks:

**Single-gameweek regression target** $y^{(1)}_{p,s,g}$:
$$
y^{(1)}_{p,s,g} = y_{p,s,g}
$$

This is the canonical FPL point score for a player in a single gameweek. It is the target for all V1 through V7 regression models, for the Optuna Safe/Aggressive specialists, and for both V5 models.

**Three-gameweek cumulative regression target** $y^{(3)}_{p,s,g}$:
$$
y^{(3)}_{p,s,g} = \sum_{k=1}^{3} y_{p,s,g+k}
$$

This is the sum of Total Points over the three gameweeks immediately following $(s,g)$. It is the target for the V6 family, used to inform transfer decisions whose horizon exceeds a single week. Missing future fixtures (e.g., end-of-season truncation) cause the corresponding sample to be dropped from training.

**Binary haul classification target** $y^{(H)}_{p,s,g}$:
$$
y^{(H)}_{p,s,g} = \mathbb{1}[y_{p,s,g} \geq \tau_H]
$$

where $\tau_H = 10$ is the haul threshold, following FPL convention that a two-digit return represents a captaincy-worthy performance. This target drives the Haul Classifier of §2.4.3, which produces the probability $\Pr[y^{(H)} = 1 \mid \mathbf{x}]$ used by the dual-engine captaincy algorithm of §2.7.

**Relationship between the three targets.** The binary target $y^{(H)}$ is a threshold function of $y^{(1)}$, and $y^{(3)}$ is a forward sum of $y^{(1)}$ values. The three therefore share substantial information content but differ in their loss function and error characteristics: regression minimises squared or absolute error symmetrically, classification minimises log-loss asymmetrically around $\tau_H$, and three-week regression averages out single-week variance at the cost of ignoring week-to-week ordering. This decomposition motivates the multi-task architecture: different FPL decisions (weekly captaincy vs. transfer planning vs. squad composition) have different optimal predictive targets.

### 2.2 Feature Taxonomy

All features derive from `master_training_set_no0min.csv` and are organised into five families. The first family is consumed unchanged from the upstream notebook; the remaining four are derived here. This separation enforces the principle that Notebook 3 is modelling-only and does not re-derive upstream quantities.

#### 2.2.1 Consumed Features (Upstream)

The following columns are consumed as produced by the preceding notebook:

| Column | Semantics | Used By |
|---|---|---|
| `Total Points` | FPL point score for the match | Target $y^{(1)}$, window generator |
| `Minutes Played` | Capped at 90 per single match | Rolling minutes, minutes-adjusted xP |
| `Position` | $\{$GK, DEF, MID, FWD$\}$ | Categorical feature in all models |
| `Next_Opponent_Difficulty` | Next fixture difficulty, $\{1, \ldots, 5\}$ | Regression models |
| `Next_Is_Home` | Binary home indicator for next fixture | Regression models |
| `Opponent Difficulty` | Current fixture difficulty | Difficulty-vs-average feature |
| `Avg_Total Points_L5`, `Avg_Minutes Played_L5` | Five-gameweek rolling averages (lagged) | Base form signal |
| `Team Total Points GW`, `Team GW Contribution Pct`, `Team Causal Contribution Pct`, `Team Contribution Rank GW` | Team-level point share features | V2 Team Share model |
| `Player Team ID`, `Player Team Name`, `Opponent ID`, `Opponent Name` | Identity / lookup | Team-based features, DGW detection |
| `Player UUID`, `Code`, `Web Name`, `Player Name` | Identity | Window indexing, lookup maps |

No re-computation of rolling averages, team contributions, or fixture difficulty occurs in Notebook 3.

#### 2.2.2 Derived Features — Windowed Lags

Given a target row for player $p$ in gameweek $(s, g)$ and window size $W \in \{4, 6, 8\}$, the lag feature vector is

$$
\boldsymbol{\ell}_{p,s,g}^{(W)} = \bigl(y_{p,s,g-W},\; y_{p,s,g-W+1},\; \ldots,\; y_{p,s,g-1}\bigr)
$$

where $y_{p,s,g-k}$ is the Total Points scored by player $p$ in gameweek $g-k$ of season $s$. Windows crossing season boundaries are discarded — we do not blend pre-summer and post-summer form. Season $s$ with fewer than $W+1$ observed gameweeks for player $p$ contributes no training windows for that player in that season.

The default window is $W=6$, which corresponds to roughly six weeks of recent form — long enough to average out single-match noise, short enough to respond to form changes. The window-size ablation of §6 validates this choice empirically.

#### 2.2.3 Derived Features — Fixture Horizon

Three fixture-difficulty features and their aggregate are computed per target row:

$$
d_{+1} = D_{p,s,g+1}, \quad d_{+2} = D_{p,s,g+2}, \quad d_{+3} = D_{p,s,g+3}
$$

where $D_{p,s,g+k}$ is the FPL-published difficulty rating (1–5) of player $p$'s opponent in gameweek $g+k$. Missing future fixtures (end of season or for pre-ingested data) are imputed as $d_{+k} = 3$ (neutral difficulty) and clipped to $[1, 5]$. The aggregate three-gameweek Fixture Difficulty Index is

$$
\text{FDI}^{(3)}_{p,s,g} = \text{clip}\bigl(d_{+1} + d_{+2} + d_{+3},\; 3,\; 15\bigr)
$$

with lower values corresponding to easier fixture runs. FDI is a primary input to the V6 three-gameweek horizon models and to the transfer advisor of §7.

#### 2.2.4 Derived Features — Reliability Metrics

Three reliability features address the fact that FPL points are noisy functions of minutes and that player volatility differs systematically across the league.

**Points Volatility (5-GW standard deviation, lagged):**
$$
\sigma^{(y)}_{p,s,g} = \text{SD}\bigl(\{y_{p,s,g-k}\}_{k=1}^{5}\bigr)
$$

High $\sigma^{(y)}$ identifies players whose point returns oscillate between blanks and hauls; low $\sigma^{(y)}$ identifies consistent contributors. Used by V3 Reliability and in the volatility–error correlation analysis of §9.2.

**Minutes Stability (5-GW rolling mean of minutes, lagged):**
$$
\bar{m}^{(5)}_{p,s,g} = \frac{1}{5} \sum_{k=1}^{5} m_{p,s,g-k}
$$

where $m_{p,s,g}$ is the (90-capped) Minutes Played in gameweek $(s,g)$. This feature distinguishes nailed starters ($\bar{m}^{(5)} \approx 90$) from rotation risks and super-subs ($\bar{m}^{(5)} < 60$). It is methodologically distinct from `Avg_Minutes Played_L5` as it is computed within-season only, preventing cross-season minute artifacts.

**Attacking Form (3-GW rolling mean of points, lagged):**
$$
\bar{y}^{(3)}_{p,s,g} = \frac{1}{3} \sum_{k=1}^{3} y_{p,s,g-k}
$$

A short-window form signal that responds faster than the 5-GW window. Complements the 5-GW rolling average by capturing recent trajectory.

#### 2.2.5 Derived Features — Expected Goal Involvement

The V7 model extends the feature set with underlying shot-quality metrics. Where xG and xA columns are directly available in the upstream data, they are used as-is. Where only ICT components (Threat, Creativity, Influence) are available, a proxy is constructed following the FPL convention of dividing Threat and Creativity by 100 to produce quantities on an xG-equivalent 0–3 range per match:

$$
\widehat{\text{xG}}_{p,s,g} = \frac{\text{Threat}_{p,s,g}}{100}, \quad \widehat{\text{xA}}_{p,s,g} = \frac{\text{Creativity}_{p,s,g}}{100}
$$

The rolling 5-gameweek expected goal involvement (lagged) is then

$$
\overline{\text{xGI}}^{(5)}_{p,s,g} = \frac{1}{5}\sum_{k=1}^{5}\bigl(\widehat{\text{xG}}_{p,s,g-k} + \widehat{\text{xA}}_{p,s,g-k}\bigr)
$$

This aggregate captures attacking contribution that is less noisy than raw Goals + Assists because it incorporates unconverted chances. The V7 ablation tests whether this lower-variance signal improves prediction over Total-Points-derived features alone.

#### 2.2.6 Derived Features — Seasonal Context

Two features provide context that a purely recent-form view misses.

**Season baseline (previous-season average points), shifted forward:**
$$
\bar{y}^{\text{prev}}_{p,s} = \frac{1}{|\{g : y_{p,s-1,g} \text{ exists}\}|} \sum_{g} y_{p,s-1,g}
$$

computed on a per-player, per-season basis from season $s-1$ and applied uniformly to all rows of player $p$ in season $s$. New players with no prior-season data receive $\bar{y}^{\text{prev}} = 0$. This feature stabilises predictions in early gameweeks when in-season form history is short.

**Within-season expanding mean against difficulty:**
$$
\bar{y}^{\text{vs}D}_{p,s,g} = \frac{1}{|\mathcal{K}|} \sum_{k \in \mathcal{K}} y_{p,s,k-1}, \quad \mathcal{K} = \{k : k < g, D_{p,s,k} = D_{p,s,g}\}
$$

For each training row, this is the mean of the player's *past points in the current season* against opponents of the same difficulty level. The lag ensures no current-match information is included. Critically, the expanding window is **within-season only** — a 2024-25 match against a difficulty-3 opponent does not inform a 2025-26 prediction against a difficulty-3 opponent. This differs from the original notebook's implementation (which was career-wide) and was adopted as a defense-ready decision to eliminate any cross-season leakage pathway. Players with no matching historical row receive a fallback value of 0.

**Season phase:**
$$
\phi_{p,s,g} = \begin{cases}
1 & \text{if } g \in [1, 10] \\
2 & \text{if } g \in [11, 19] \\
3 & \text{if } g \in [20, 29] \\
4 & \text{if } g \in [30, 38]
\end{cases}
$$

A piecewise-constant calendar feature. Phase 1 captures early-season freshness, 2 captures first-rotation dynamics, 3 captures winter-congestion fixture density, and 4 captures end-of-season high-stakes motivation.

#### 2.2.7 Feature Vector

The full feature vector for the single-gameweek regression task at window size $W$ is

$$
\mathbf{x}^{(1,W)}_{p,s,g} = \bigl(\boldsymbol{\ell}^{(W)}, \text{Position}, d_{+1}, \mathbb{1}[\text{Home}_{+1}], \bar{m}^{(5)}, \phi, \bar{y}^{\text{prev}}, \bar{y}^{\text{vs}D}\bigr)
$$

yielding $W + 7$ dimensions (categorical `Position` is treated natively by LightGBM's categorical split). Variants extend this base with model-specific additions: V2 adds the team share features; V3 adds $\sigma^{(y)}$, $\bar{m}^{(5)}$ (already present), $\bar{y}^{(3)}$, and the team share; V7 adds $\overline{\text{xGI}}^{(5)}$; V6 adds $d_{+2}$, $d_{+3}$, and $\text{FDI}^{(3)}$.

### 2.3 Temporal Cross-Validation Strategy

FPL is a temporal system: gameweek $g$ in season $s$ is strictly earlier than gameweek $g+1$, and any validation protocol that permits later gameweeks to influence predictions for earlier gameweeks induces look-ahead bias. We therefore adopt a **fixed-window temporal holdout** rather than k-fold cross-validation or random splitting.

**Formal specification.** Let $T_{\text{val}}$ be the validation gameweek window, defined as the final $W_{\text{val}}$ completed gameweeks of the current season. Let $g^{\star}$ be the most recent gameweek with complete data (i.e., $\max\{g : y_{p,s_{\text{curr}},g} \text{ exists for at least one player}\}$). Then

$$
T_{\text{val}} = \{(s_{\text{curr}}, g) : g^{\star} - W_{\text{val}} + 1 \leq g \leq g^{\star}\}
$$

with default $W_{\text{val}} = 5$. The training set comprises all rows *not* in $T_{\text{val}}$: all historical seasons in full, plus the current season up to and including $g^{\star} - W_{\text{val}}$.

**Automatic gameweek detection.** The current gameweek and validation window are not hardcoded. At notebook load time, $g^{\star}$ is read from the data as $\max_{p,g}\{g : (s_{\text{curr}}, g) \in \text{data}\}$, and $T_{\text{val}}$ is computed dynamically. This ensures the notebook produces valid results when re-run on future data without manual configuration.

**The current-gameweek prediction.** Beyond validation, the notebook also produces live predictions for gameweek $g^{\star} + 1$. These are out-of-sample in the strictest sense — no data from $g^{\star} + 1$ exists at model training time.

**Justification for fixed-window over rolling-origin.** We considered rolling-origin cross-validation (train on GW1 through GW $k$, validate on GW $k+1$; repeat for $k = 6, \ldots, g^{\star}-1$) but rejected it on three grounds. First, the FPL season is non-stationary — player form, team form, and fixture difficulty all evolve as the season progresses, so a single validation window matches deployment conditions more faithfully than averaging over a full season's worth of one-step-ahead errors. Second, rolling-origin massively inflates compute cost (approximately 30× for a full-season sweep) without producing proportionally richer diagnostic signal. Third, the published FPL literature [^amosnjr_2020][^boo_2022] converges on fixed-window temporal holdout as the standard protocol, and thesis defensibility benefits from methodological alignment with established practice.

**Group structure for ranking models.** For the LambdaRank model of §2.4.2, the training data requires a query-group structure: within each query group, the model learns to rank items relative to each other. We use $(s, g)$ as the grouping variable — one query per (season, gameweek) pair — so the model learns to rank all players appearing in the same gameweek against each other. This matches the deployment use case: at any given gameweek, the user needs an ordering over the available player pool.

[^amosnjr_2020]: M. Amosnjr, "Mathematical Modelling for Fantasy Premier League," MSc Thesis, University of Manchester, 2020.

[^boo_2022]: S. H. Boo, "Machine Learning Approaches to Fantasy Premier League Team Selection," *Expert Systems with Applications*, vol. 190, 2022.

### 2.4 Core Model Specifications

Eight models receive full methodological treatment in this section. Each is specified by its loss function, feature set, hyperparameters, and modelling rationale. Implementation code appears in §5; implementation-level concerns (categorical handling, early-stopping protocols, persistence format) are deferred to that section.

All gradient-boosted tree models use the LightGBM library [^lightgbm], which implements the histogram-based gradient boosting variant with leaf-wise tree growth. Categorical features are handled via LightGBM's native categorical split rather than one-hot encoding, which matches Fisher's optimal partition for the Gini impurity criterion in $O(k \log k)$ time.

[^lightgbm]: G. Ke *et al.*, "LightGBM: A Highly Efficient Gradient Boosting Decision Tree," in *Advances in Neural Information Processing Systems 30*, 2017, pp. 3146–3154.

#### 2.4.1 Model V1 — Baseline Regression

**Task:** Single-gameweek point regression.

**Loss:** L2 (squared error), minimised by LightGBM's gradient-boosted ensemble.

**Feature set:** $\mathbf{x}^{(1,6)}$ as defined in §2.2.7, dimension 13.

**Hyperparameters:**

| Parameter | Value | Rationale |
|---|---|---|
| `n_estimators` | 1500 | Upper bound; early stopping selects actual depth |
| `learning_rate` | 0.01 | Shallow rate for stable convergence |
| `max_depth` | 7 | Balances bias–variance for 13-feature input |
| `num_leaves` | 40 | Allows complex interactions; bounded below $2^{\text{max\_depth}}$ |
| `boosting_type` | `gbdt` | Standard gradient boosting |
| `random_state` | 42 | Reproducibility |

Early stopping on validation MAE with patience 100 rounds. The model is the baseline against which all other architectures are benchmarked.

#### 2.4.2 Model: LambdaRank Blender

**Task:** Single-gameweek relevance ranking (query-grouped).

**Loss:** LambdaRank objective [^lambdarank], a listwise ranking loss that maximises discounted cumulative gain (DCG) via pairwise gradient approximations.

**Relevance label binning:** Continuous Total Points are discretised into six relevance levels:

| Bin | Points Range | Label |
|---|---|---|
| Blank | $(-\infty, 0]$ | 0 |
| Low | $[1, 2]$ | 1 |
| OK | $[3, 4]$ | 2 |
| Good | $[5, 7]$ | 3 |
| Great | $[8, 11]$ | 4 |
| Haul | $[12, \infty)$ | 5 |

with `label_gain = [0, 1, 2, 4, 8, 16]` — an exponentially increasing gain schedule that rewards correct ordering of high-ceiling players more than correct ordering of blanks.

**Query structure:** $(s, g)$ pairs, sorted chronologically. Each group corresponds to a single gameweek of all active players.

**Feature set:** $\mathbf{x}^{(1,6)}$, identical to V1 for direct blending compatibility.

**Hyperparameters:** $n_\text{estimators}=1500$, learning rate 0.01, max_depth 6, num_leaves 31, random_state 42.

**Blended prediction.** At inference time, V1's regression output and the LambdaRank relevance score are min-max normalised to $[0, 1]$ and combined:

$$
\hat{y}^{\text{blended}} = \left[\alpha \cdot \text{MinMax}(\hat{y}^{\text{V1}}) + (1-\alpha) \cdot \text{MinMax}(\hat{r}^{\text{rank}})\right] \cdot (y_{\max} - y_{\min}) + y_{\min}
$$

with $\alpha = 0.6$. The final rescaling maps the blended $[0,1]$ score back to the point-value range of the validation set. The 60/40 weighting was empirically determined: $\alpha < 0.5$ produces rankings that outperform raw regression on Spearman correlation but degrade MAE excessively; $\alpha > 0.7$ effectively discards the ranking signal. The blend improves ceiling detection (high-scoring player ordering) at a small cost in MAE, a trade-off validated in the stratified analysis of §9.1.

[^lambdarank]: C. Burges, "From RankNet to LambdaRank to LambdaMART: An Overview," Microsoft Research Technical Report MSR-TR-2010-82, 2010.

#### 2.4.3 Model: Haul Classifier

**Task:** Binary haul classification with $y^{(H)} = \mathbb{1}[y \geq 10]$.

**Loss:** Binary cross-entropy with class-imbalance correction (`scale_pos_weight = 3.0`). Hauls are approximately 3–5% of all player-gameweek observations, so without re-weighting the classifier converges to predicting "no haul" universally.

**Feature set (extended, 13 features):**

1. Six lags $\boldsymbol{\ell}^{(6)}$
2. `Position` (categorical)
3. `Next_Opponent_Difficulty`
4. `Next_Is_Home`
5. `Rolling_Avg_Minutes_5`
6. `Season_Phase`
7. `Prev_Season_Avg_Points`
8. **`recent_haul_flag`** = $\mathbb{1}[\exists k \in \{1,2,3\} : y_{p,s,g-k} \geq 10]$
9. **`lag_max_6`** = $\max_{k \in \{1,\ldots,6\}} y_{p,s,g-k}$
10. **`is_elite`** = $\mathbb{1}[\bar{y}_{p,s} > 4.5]$
11. **`fixture_ease`** = $6 - d_{+1}$
12. **`ceiling_ratio`** = `lag_max_6` / 10
13. $\bar{y}^{\text{vs}D}$

Features 8–12 are classifier-specific and designed to capture ceiling propensity rather than expected value. `recent_haul_flag` tests the hypothesis that haul-prone players cluster their hauls; `lag_max_6` captures maximum recent output; `ceiling_ratio` normalises that maximum against the haul threshold.

**Hyperparameters:** `objective=binary`, `n_estimators=500`, `learning_rate=0.05`, `max_depth=6`, `num_leaves=31`, `scale_pos_weight=3.0`, `random_state=42`.

**Decision threshold:** Probability threshold $\tau_D = 0.35$, below the naive 0.5 to boost recall given the asymmetric cost of missing a haul (a missed captaincy opportunity costs more expected points than a false positive costs predictive accuracy). The threshold–precision–recall trade-off is characterised in §5.3.

**Interpretation.** The classifier does not replace the regression — it complements it. A player with moderate expected value (V1 regression of 4.5 points) but high haul probability (classifier of 0.6) is a strong captaincy candidate whose ceiling the regression underestimates. The dual-engine captaincy algorithm of §2.7 formalises this complementarity.

#### 2.4.4 Model V3 — Reliability Engine

**Task:** Single-gameweek point regression with explicit penalisation of overfitting to volatile players.

**Feature set (15 features):** Base V1 features plus

- `Points_Volatility_5` ($\sigma^{(y)}$): five-gameweek SD of points
- `Min_Stability_5` ($\bar{m}^{(5)}$): five-gameweek rolling mean of minutes
- `Attacking_Form_3` ($\bar{y}^{(3)}$): three-gameweek rolling mean of points
- `Team_Point_Share`: consumed from upstream

**Hyperparameters:** Identical to V1 except regularisation is enabled via `reg_alpha=0.3`, `reg_lambda=0.3`. L1 (alpha) drives sparse feature selection; L2 (lambda) shrinks coefficient magnitudes toward zero. The matched regularisation strengths reflect a conservative regularisation regime; stronger values over-smoothed on early pilots.

**Modelling rationale.** FPL point returns for volatile players are poorly predicted by point estimates. A volatile player's "true" one-step-ahead distribution is bimodal (frequent blanks punctuated by occasional hauls); MAE-minimising regression of a bimodal distribution converges to the median, which is often zero. The reliability engine addresses this two ways: first, by adding $\sigma^{(y)}$ as a direct input (so the model can learn "when this player's volatility is high, shrink predictions toward a safer baseline"); second, via L1/L2 regularisation that discourages the model from exploiting spurious feature interactions learned on noisy volatile-player data. The result is a model that is slightly worse in absolute MAE but better-calibrated across the scoring distribution, as validated in §9.1.

#### 2.4.5 Model V4 — Elite DART with Sample Weighting

**Task:** Single-gameweek point regression with bias correction toward elite-player patterns.

**Boosting type:** Dropouts meet Multiple Additive Regression Trees (DART) [^dart]. At each boosting round, DART randomly drops a fraction of previously-built trees before fitting the new one, preventing any single tree from dominating the ensemble. This produces better generalisation than standard GBDT on noisy targets.

**Sample weighting:** Training rows where $\bar{y}_{p,s} > 4.5$ (elite players) are weighted 2.5×; all other rows are weighted 1.0. This biases the fit toward the patterns exhibited by consistent high scorers — whose predictions are disproportionately important for captaincy, transfers, and squad allocation.

**Feature set:** $\mathbf{x}^{(1,6)}$, identical to V1.

**Hyperparameters:**

| Parameter | Value | Rationale |
|---|---|---|
| `boosting_type` | `dart` | Dropout boosting |
| `drop_rate` | 0.1 | 10% of trees dropped per round |
| `n_estimators` | 1200 | Fixed — DART does not support early stopping |
| `learning_rate` | 0.05 | Higher than V1 since DART dropout dampens aggressive updates |
| `max_depth` | 7 | Matches V1 |
| `reg_alpha` | 0.5 | Stronger than V3 to counter elite-bias |
| `reg_lambda` | 0.5 | Same |
| `random_state` | 42 | — |

**Methodological note on early stopping.** DART is incompatible with early stopping because the random tree dropping causes the validation loss to fluctuate non-monotonically. This is a genuine empirical discovery documented in the LightGBM project issues [^lightgbm_dart_es] and confirmed in our pilot training runs. We therefore train for the full 1200 rounds without early stopping and rely on the regularisation to prevent overfitting. This decision is defense-documented.

**Modelling rationale.** V4 targets two distinct pathologies simultaneously. DART addresses overfitting to outliers; elite weighting addresses the inverse of the standard ML bias — rather than upweighting minority classes, we upweight the patterns that matter for deployment. The combination is intentional: elite-only training (V5.2) produces low-variance models that miss ceiling events, and pure DART without weighting (a variant we piloted) gives insufficient attention to the players actually drafted. V4 is the resulting compromise, empirically the best model by Elite MAE in §9.

[^dart]: K. V. Rashmi and R. Gilad-Bachrach, "DART: Dropouts meet Multiple Additive Regression Trees," in *Proc. 18th Int. Conf. on Artificial Intelligence and Statistics*, 2015, pp. 489–497.

[^lightgbm_dart_es]: LightGBM GitHub issue #1893, "Early stopping incompatible with DART boosting," 2018.

#### 2.4.6 Model V7 — Expected Goal Involvement Extension

**Task:** Single-gameweek point regression augmented with underlying xG/xA signal.

**Feature set (14 features):** $\mathbf{x}^{(1,6)}$ plus $\overline{\text{xGI}}^{(5)}$ (five-GW rolling expected goal involvement, lagged).

**Hyperparameters:** Identical to V4 (DART boosting, drop_rate 0.1, elite weighting 2.5×). The choice of V4 as the base architecture (rather than V1) ensures the xG/xA ablation tests the *additive* value of underlying metrics rather than the value of underlying metrics *instead of* DART+weighting.

**Modelling rationale.** Total Points is a noisy function of shot volume, shot quality, and finishing conversion. A forward who takes six shots worth a combined 1.4 xG but converts none is invisible to a points-based regression; but that pattern is predictive of future scoring output. Including $\overline{\text{xGI}}^{(5)}$ makes this latent quality visible. If the upstream data includes official xG/xA columns, they are used directly; otherwise, the Threat/100 and Creativity/100 proxies of §2.2.5 substitute. The V7 ablation quantifies whether this substitution captures enough underlying signal to justify its computational cost.

#### 2.4.7 Model: Optuna Safe Specialist

**Task:** Single-gameweek point regression with hyperparameters tuned to minimise validation MAE.

**Optimisation method:** Tree-structured Parzen Estimator (TPE) [^tpe] via Optuna, 50 trials. The TPE sampler models the conditional distribution of high-performing hyperparameters and preferentially samples from that region, converging faster than random search or grid search on continuous spaces.

**Objective:** Pure MAE minimisation on the validation set:
$$
\theta^{\star}_{\text{safe}} = \arg\min_{\theta} \frac{1}{|T_{\text{val}}|} \sum_{(p,s,g) \in T_{\text{val}}} |y_{p,s,g} - \hat{y}_{\theta}(\mathbf{x}_{p,s,g})|
$$

**Search space (LightGBM):**

| Parameter | Range | Scale |
|---|---|---|
| `n_estimators` | [300, 800] | integer |
| `learning_rate` | [0.005, 0.1] | log |
| `max_depth` | [3, 9] | integer |
| `num_leaves` | [15, 127] | integer |
| `min_child_samples` | [10, 100] | integer |
| `reg_alpha` | [1e-4, 2.0] | log |
| `reg_lambda` | [1e-4, 2.0] | log |
| `subsample` | [0.5, 1.0] | uniform |
| `colsample_bytree` | [0.5, 1.0] | uniform |

**Feature set:** $\mathbf{x}^{(1,6)}$, identical to V1.

**Modelling rationale.** V1 uses hand-tuned hyperparameters; Optuna Safe tests whether systematic search finds meaningfully different settings. The narrow MAE band observed across trials (validated in §5.7) is itself a methodological result — it confirms V1 is near the achievable MAE ceiling given the feature set, and that additional predictive gains require feature engineering, not hyperparameter optimisation.

#### 2.4.8 Model: Optuna Aggressive Specialist

**Task:** Single-gameweek point regression with hyperparameters tuned to maximise precision on the high-scoring tier.

**Objective:** Layer-3 Precision (precision on predictions of $\hat{y} \geq 8$ points):
$$
\theta^{\star}_{\text{agg}} = \arg\max_{\theta} \left[\text{Precision}(\hat{y}_{\theta} \geq 8, y \geq 8) - 0.1 \cdot \max(0, \text{MAE} - 4.0)\right]
$$

The precision term is the primary objective; the MAE penalty prevents degenerate solutions that predict "8+" for every player. 50 TPE trials.

**Additional search dimension — elite weighting:** Unlike Safe, Aggressive includes the sample-weighting multiplier as a tunable parameter:
$$
w_i = \begin{cases} w^{\text{elite}} & \text{if } y_{p,s,g} \geq 8 \\ 1.0 & \text{otherwise} \end{cases}
$$
with $w^{\text{elite}} \in [1.0, 5.0]$ searched over. This parameterises the trade-off between base distribution fit and high-ceiling emphasis.

**Feature set:** $\mathbf{x}^{(1,6)}$, identical to V1.

**Modelling rationale.** A naïve Optuna MAE-optimisation (the Safe specialist) produces a model that is biased against predicting haul-range outcomes because those are rare and the loss function does not incentivise recall of them. Aggressive explicitly asks: "what hyperparameters produce a model where, *when it does commit to a high-scoring prediction, it is right most of the time*?" The result is a low-recall but high-precision predictor that complements the Haul Classifier: where the classifier produces a probability, the Aggressive regressor produces a point estimate calibrated specifically against the captaincy decision threshold.

[^tpe]: J. Bergstra *et al.*, "Algorithms for Hyper-Parameter Optimization," in *Advances in Neural Information Processing Systems 24*, 2011, pp. 2546–2554.

### 2.5 Ablation Model Specifications

Nine additional models are trained and reported as ablation studies. Each tests a specific methodological question. Specifications are concise; full hyperparameters appear with the corresponding implementation in §6.

| Model | Task | Key Variation | Research Question |
|---|---|---|---|
| V1 Scaled | 1-GW regression | V1 with StandardScaler on numeric features | Does LightGBM benefit from feature scaling? (Expected: no.) |
| V2 Team Share | 1-GW regression | V1 + `Team_Point_Share` feature | Does talisman-status information improve prediction? |
| V5.1 Mass | 1-GW regression | V1 trained on full data, evaluated on elites | Does elite-only evaluation of a general model reveal weaknesses? |
| V5.2 Elite Specialist | 1-GW regression | V1 trained on $\bar{y}_{p,s} > 4.5$ subset only | Does specialising training improve elite prediction? |
| V6 1-GW Horizon | 1-GW regression | V1 + FDI + `Opponent_Diff_plus_{2,3}` | Does multi-GW fixture context help next-GW prediction? |
| V6 Standard 3-GW | 3-GW regression | Target = 3-GW sum, base features | Can LightGBM forecast multi-week sums? |
| V6 Reliability 3-GW | 3-GW regression | V6 Standard + reliability features | Does reliability context help 3-GW forecasts? |
| V6 DART 3-GW | 3-GW regression | V6 Standard + DART + elite weighting | Do V4 innovations transfer to the 3-GW task? |
| Window-4w / Window-8w | 1-GW regression | Same features, different lag window | What is the optimal lookback? |

Each ablation is reported in the master audit ledger of §9 alongside the core models, providing a unified comparison. Ablation-specific discussion appears in §6.

### 2.6 Portfolio Optimisation Frameworks

Two portfolio construction methods are implemented: a heuristic **relief-score optimizer** (the production engine) and a mathematically optimal **Markowitz mean-variance MILP formulation**. Both respect the full FPL rule set. Both consume model predictions from §2.4; they differ in search strategy and in whether they model risk explicitly.

#### 2.6.1 Common Formulation

Let $\mathcal{P}$ be the pool of eligible players (after availability filtering), and for each $p \in \mathcal{P}$ let $c_p$ denote price (in £m), $\pi_p$ denote position $\in \{\text{GK}, \text{DEF}, \text{MID}, \text{FWD}\}$, $t_p$ denote Premier League club, and $\hat{y}_p$ denote predicted expected points for the target gameweek.

The squad selection variable is $x_p \in \{0, 1\}$ indicating whether player $p$ is selected. Any feasible squad must satisfy:

- **Squad size:** $\sum_{p} x_p = 15$
- **Budget:** $\sum_{p} c_p \cdot x_p \leq 100.0$
- **Position quotas:** $\sum_{p : \pi_p = \text{GK}} x_p = 2$, $= 5$ for DEF, $= 5$ for MID, $= 3$ for FWD
- **Per-club cap:** $\sum_{p : t_p = T} x_p \leq 3$ for every club $T$

#### 2.6.2 Relief-Score Heuristic (Strategy B — Production Engine)

**Algorithm:** Greedy construction with iterative local search.

1. Select three low-cost "dead fodder" bench players (one each of bench GK, bench DEF, bench MID) with chance of playing ≥ 75%. These are structurally required by the 15-man squad format but add minimal expected value.
2. For the remaining 12 "core" slots, greedily select players by blended score $s_p = \hat{y}_p \cdot (1 + H_p/50)$, where $H_p$ is the haul probability from the Haul Classifier (§2.4.3). The $/50$ normalisation gives haul-eligible players a material but bounded boost.
3. If total cost exceeds budget, iteratively **downgrade** the player with highest *relief score* $r_p = (c_p - c_p^{\text{repl}}) / (\hat{y}_p + 0.1)$ (largest cost drop per unit of expected-point sacrifice) to the cheapest same-position player not currently selected.
4. If total cost is under budget by > £0.2m, iteratively **upgrade** the weakest-score core player to a better same-position player within the remaining headroom, subject to per-club and formation constraints.
5. For the starting eleven, pick the highest-scoring GK, three DEFs, one FWD, and six additional outfield players from among the 12 core players. A formation safety guard ensures at least 3 DEFs and 1 FWD survive the selection.
6. For captaincy, apply the dual-engine algorithm of §2.7.

**Practical enhancements.** The production engine further includes: **DGW enforcement** (at least 8 of 12 core players should be from DGW teams when DGWs are detected), **penalty-taker boosting** ($\hat{y}_p$ multiplied by 1.15 for known penalty takers), **bench upgrade phase** (any residual budget improves bench fodder quality), and **availability filtering** (players flagged as injured/suspended are removed from $\mathcal{P}$ before selection).

**Optimality guarantee.** None. The algorithm is a fast local search and may converge to a local rather than global optimum. In practice, the DGW enforcement step can force the upgrade loop into oscillation, which is prevented by a maximum-iteration guard (100 iterations) and an upgraded-player tracking set.

#### 2.6.3 Markowitz MILP (Mean-Variance Efficient Frontier)

**Objective:** Given a risk-aversion parameter $\lambda \geq 0$, maximise the utility function
$$
U_\lambda(\mathbf{x}) = \sum_{p} \bigl[\hat{y}_p \cdot \alpha_p \cdot \beta_p + \gamma \cdot \hat{y}_p \cdot \hat{H}_p - \lambda \cdot \sigma^{(y)}_p\bigr] \cdot x_p
$$

where

- $\alpha_p = 1 + \epsilon_{\text{elite}} \cdot \mathbb{1}[\bar{y}_p > 4.5]$ is the elite multiplier
- $\beta_p = 1 + \epsilon_{\text{DGW}} \cdot \mathbb{1}[t_p \in \text{DGW teams}]$ is the DGW multiplier
- $\hat{H}_p \in [0,1]$ is the normalised haul probability
- $\gamma$ is the haul-boost weight
- $\sigma^{(y)}_p$ is the player's five-GW point volatility (§2.2.4)

**Strategy configurations:**

| Strategy | $\lambda$ | $\gamma$ | $\epsilon_{\text{elite}}$ | $\epsilon_{\text{DGW}}$ |
|---|---|---|---|---|
| Safe | 0.5 | 0.0 | 0.05 | 0.10 |
| Balanced | 0.2 | 0.15 | 0.10 | 0.20 |
| Aggressive | 0.0 | 0.40 | 0.15 | 0.30 |

**Constraints:** As in §2.6.1. Price is represented as an integer (×10 scaling) to eliminate floating-point drift in the budget constraint. The formation legality of the starting eleven is enforced post-solve by a deterministic selection algorithm, not as part of the MILP.

**Solver:** CBC (Coin-or branch and cut) via PuLP [^pulp], with a 60-second time limit and 0.1% relative optimality gap.

**Efficient frontier.** By sweeping $\lambda$ from 0 (Aggressive) to 1 (Safe-maximal), the MILP produces an efficient frontier of Pareto-optimal squads. Due to FPL's tight integer constraints (discrete squad + per-club limits), the frontier tends to collapse to a small number of distinct solutions rather than a continuous curve — a methodological finding in itself, documented in §7.2.

**Optimality guarantee.** For each $\lambda$ value, the MILP produces a provably optimal squad (within the CBC gap tolerance) given the stated utility function. The claim is *conditional on the utility specification*, not an absolute statement — a different $\lambda, \gamma, \epsilon$ choice produces a different optimum.

#### 2.6.4 Relative Positioning

| Property | Heuristic | MILP |
|---|---|---|
| Optimality | Local | Global (given utility spec) |
| Risk modelling | Implicit (captaincy only) | Explicit ($\lambda \cdot \sigma^{(y)}$) |
| DGW enforcement | Hard target (≥8 DGW players) | Soft via $\beta_p$ multiplier |
| Runtime | Fast (seconds) | Medium (10–60 seconds per $\lambda$) |
| Interpretability | Step-by-step traceable | Utility-based |

Both engines are retained and compared in the backtest of §8.

[^pulp]: J.-S. Roy and S. Mitchell, "PuLP: A Linear Programming Toolkit for Python," COIN-OR Foundation, 2007.

### 2.7 Auto-Substitution and Captaincy Algorithms

Both algorithms are validated operational code from the original pipeline and are preserved here with formal specification. They govern how predictions translate into actual FPL points.

#### 2.7.1 Captaincy — Dual-Engine Algorithm

**Inputs.** The starting eleven $\mathcal{S} \subset \mathcal{P}$ with $|\mathcal{S}| = 11$. For each $p \in \mathcal{S}$, predicted points $\hat{y}_p$, haul probability $\hat{H}_p \in [0, 100]$ (in percent), elite flag $E_p = \mathbb{1}[\bar{y}_p > 4.5]$, DGW flag $G_p = \mathbb{1}[t_p \in \text{DGW teams}]$.

**Haul cap.** To prevent extreme haul probabilities from dominating captaincy selection, the haul signal is capped at 35%:
$$
\tilde{H}_p = \min(\hat{H}_p, 35)
$$

This cap is empirically derived — above ~35% haul probability, the classifier is observed to over-commit on a small number of high-form players, and uncapped use produces captaincy selections that a human FPL analyst would regard as anomalously aggressive.

**Captain Expected Value score:**
$$
\text{CEV}_p = \hat{y}_p \cdot \left(1 + \frac{\tilde{H}_p}{50} + 0.3 \cdot G_p\right)
$$

Three components: base expected value, haul-probability bonus (damped by /50), and a 30% structural bonus for DGW players who play two fixtures.

**Elite multiplier (final):**
$$
\text{Score}_p = \text{CEV}_p \cdot (1 + 0.1 \cdot E_p)
$$

A modest 10% elite bonus acknowledges that elite players' point distributions have thicker upper tails than classifier probabilities alone capture.

**Selection:**
- **Captain:** $p^{\star} = \arg\max_{p \in \mathcal{S}} \text{Score}_p$
- **Vice-captain:** $p^{\star\star} = \arg\max_{p \in \mathcal{S} \setminus \{p^{\star}\}} \text{Score}_p$

Both must be nominated from the starting eleven (not the bench) per FPL rules.

#### 2.7.2 Automatic Substitution

**Scope.** Applied after a gameweek completes, modifying the scoring eleven based on actual minutes played.

**FPL rule specification.** If a starter played zero minutes, their bench substitute — ordered by FPL-defined priority — is inserted, provided the resulting eleven satisfies legal formation constraints (at least 1 GK, at least 3 DEF, at least 1 FWD).

**Bench priority order.** Outfield substitutes are ordered by user-nominated priority (1 → 2 → 3); the bench goalkeeper is reserved exclusively for replacing a non-playing starting goalkeeper.

**Algorithm.** Let $\mathcal{S} = (s_1, \ldots, s_{11})$ be the starting eleven, $\mathcal{B} = (b_1, b_2, b_3, b_{\text{GK}})$ the bench (priority order), and $m_p$ the minutes played for player $p$. For each pass (up to 11 total to handle cascading substitutions):
for each starter s in S (in original position order):
if m(s) > 0:
continue  (starter played, no sub needed)
if position(s) == GK:
    if m(b_GK) > 0:
        replace s with b_GK; mark b_GK used

else (outfield starter DNP):
    for each available bench outfield sub b in priority order:
        if m(b) == 0:
            continue (sub also DNP, try next)
        
        tentative_XI = S with s replaced by b
        if tentative_XI has >= 3 DEF and >= 1 FWD and >= 1 GK:
            apply substitution; mark b used
            break

Additional passes run until no substitution occurs in a pass.

**Captain-DNP special rule.** If the captain $p^{\star}$ plays zero minutes, the vice-captain's score is doubled instead. This is handled after substitutions are applied: if $m(p^{\star}) = 0$, the scoring contribution of $p^{\star\star}$ is $2 y_{p^{\star\star}}$ (regardless of whether $p^{\star}$ or $p^{\star\star}$ ended up in the final eleven via substitution).

**Validation.** The algorithm is directly transcribed from the original (user-validated) implementation, with formal correctness verified by the constraint-checking unit tests in the backtest harness of §8. The algorithm is preserved verbatim; no "simplification" has been applied because the explicit position constraints and bench-priority rules are what make the result FPL-rule-compliant.

### 2.8 Evaluation Metrics

Predictive performance is reported across multiple metrics chosen to expose different failure modes. No single metric suffices because different deployment decisions (squad selection vs. captaincy vs. transfer planning) have different cost asymmetries.

#### 2.8.1 Point Regression Metrics

**Mean Absolute Error (MAE):**
$$
\text{MAE} = \frac{1}{|T_{\text{val}}|} \sum_{(p,s,g) \in T_{\text{val}}} |y_{p,s,g} - \hat{y}_{p,s,g}|
$$

The primary metric for all point regression models. Measured in raw FPL points. Preferred over RMSE for this domain because FPL returns are non-Gaussian (a 15-point haul is genuinely rare, not an "outlier" to be down-weighted).

**Root Mean Squared Error (RMSE):**
$$
\text{RMSE} = \sqrt{\frac{1}{|T_{\text{val}}|} \sum (y - \hat{y})^2}
$$

Reported alongside MAE for completeness. Higher RMSE than MAE indicates the model is making large individual errors (consistent with ceiling underestimation).

**Signed Error (Bias):**
$$
\text{Bias} = \frac{1}{|T_{\text{val}}|} \sum (\hat{y} - y)
$$

Positive bias = the model systematically overpredicts; negative = underpredicts. Reported per scoring tier to characterise the distribution of errors, not just their magnitude.

#### 2.8.2 Stratified MAE

A single aggregate MAE masks tier-specific performance differences. We report MAE decomposed by the player's actual score:

| Tier | Actual Points | Deployment Relevance |
|---|---|---|
| Blanks | $[0, 2]$ | Low — little information |
| Low | $[3, 5]$ | Moderate — mid-table squad picks |
| Mid | $[6, 9]$ | High — prime captaincy candidates |
| Haul | $[10, \infty)$ | Highest — the ceiling the model must capture |

The Haul-tier MAE is the most actionable single metric for captaincy decisions; the Blank-tier MAE is the least interesting (most models can predict zero). §9 reports all four for every model.

#### 2.8.3 Ranking Metrics

**Spearman Rank Correlation:**
$$
\rho = \frac{\sum_{i} (r_i^y - \bar{r}^y)(r_i^{\hat{y}} - \bar{r}^{\hat{y}})}{\sigma_{r^y} \sigma_{r^{\hat{y}}}}
$$

where $r_i^y$ and $r_i^{\hat{y}}$ are the ranks of actual and predicted points. Spearman captures whether the model orders players correctly *independent of absolute error*. A model with terrible MAE but high Spearman is still useful for squad selection (rank-based picks); a model with low MAE and low Spearman implies systematic errors that correlate across players.

#### 2.8.4 Classification Metrics

For the Haul Classifier (§2.4.3), standard binary metrics:

- **Precision** $= \text{TP}/(\text{TP}+\text{FP})$: when the model predicts haul, how often is it right?
- **Recall** $= \text{TP}/(\text{TP}+\text{FN})$: of actual hauls, what fraction did the model identify?
- **F1** $= 2 \cdot \text{Precision} \cdot \text{Recall} / (\text{Precision} + \text{Recall})$: harmonic mean
- **ROC-AUC:** area under receiver operating characteristic, threshold-independent

For captaincy decisions, **precision is the critical metric** — a false positive ("this player will haul") causes a wrong captaincy pick, a cost of several points; a false negative merely passes on a captaincy opportunity, which is less costly.

#### 2.8.5 Volatility–Error Correlation

The correlation between a player's point volatility $\sigma^{(y)}_p$ and the absolute prediction error $|y - \hat{y}|$ is reported and visualised in §9.2. Positive correlation (expected) confirms that volatile players are harder to predict; this is used to calibrate risk-aware deployment in §7.

#### 2.8.6 Backtest Metrics

The 10-GW backtest (§8) reports per-gameweek and cumulative actual points for each strategy (Safe, Balanced, Aggressive from the Markowitz MILP), with captain auto-sub and vice-captain replacement applied. Additional metrics include:

- **Strategy ranking:** which strategy produced the highest cumulative points
- **Model calibration ratio:** $\text{Ratio} = \sum \text{Actual} / \sum \text{Predicted xP}$ — the empirically observed multiplier to translate raw model predictions into realistic expectations
- **Average auto-subs per GW:** how often the auto-sub rule fires, indirectly measuring rotation/minute risk in the selected squads
- **Captain hit rate:** fraction of GWs where the selected captain played > 0 minutes

Calibration ratio is the most important of these for honest deployment. We report it as-is rather than "fixing" model predictions to match actuals, following the principle that the model should be honest about its biases, not silently corrected.

## 3. Environment and Reproducibility

This section specifies the computational environment, random seeds, and persistence conventions. Reproducibility at this level is required for thesis-grade work: re-running this notebook with the same inputs must produce bit-identical model artifacts, metrics, and optimiser outputs.

### 3.1 Seed Management

All stochastic operations use a fixed random seed of 42:

- LightGBM model training (`random_state=42`)
- Optuna hyperparameter search (`TPESampler(seed=42)`)
- NumPy operations that invoke the default RNG
- Train/validation split (deterministic; no randomness involved — fixed gameweek window)

### 3.2 Persistence Conventions

Model artifacts and intermediate outputs are written to a structured output directory:
output/
└── forecaster_pro/
├── models/              # Trained LightGBM boosters (.txt)
├── predictions/         # Leaderboards and forecasts (.csv)
├── metrics/             # Stratified MAE, audit ledgers (.csv)
└── figures/             # Diagnostic plots (.png)
All CSV writes use atomic rename (write to `.tmp`, then rename) to prevent partial-file corruption in the event of a process interruption.

### 3.3 Dependency Versions

Versions were frozen at the time of thesis submission. Minor version drift in pure-Python dependencies is acceptable; major version changes in LightGBM, scikit-learn, or PuLP require re-validation of the stratified MAE results.